# 🤖 تدريب الموديل المحاسبي الذكي (AI Accountant v2)

هذا الملف مجهز للعمل مباشرة على **Google Colab**.
تأكد أولاً من تفعيل **T4 GPU** من خلال القائمة العلوية:
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

**خطوة هامة قبل البدء:** قم برفع ملف `dataset.jsonl` إلى Colab من خلال سحبه وإفلاته في أيقونة الملفات 📁 على اليسار.

In [ ]:
# 1. تثبيت مكتبة unsloth والمكتبات المساعدة للتدريب السريع
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.15.0" peft transformers datasets

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 2. تحديد الموديل الصغير (تم تغيير الموديل إلى Qwen2.5-3B بحجم لا يتعدى 2 جيجا للعمل بسرعة فائقة)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. إعداد الـ Fine-Tuning الذكي (LoRA) عشان نعدل الأوزان بدون ما نكبر حجم الموديل
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 4. تحميل ملف الـ Dataset
dataset = load_dataset("json", data_files="dataset.jsonl")

# دالة لتنسيق النص عشان الموديل يفهم السؤال والجواب بشكل صحيح
def format_prompts(batch):
    texts = []
    for prompt, response in zip(batch["prompt"], batch["response"]):
        text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

In [ ]:
# التحقق من تحميل البيانات بشكل صحيح (عرض أول 5 أمثلة)
print(dataset["train"].select(range(5)))

In [ ]:
# 5. بدء التدريب (التنفيذ)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # تقدر تزودها لـ 120 أو 200 لو عايز دقة أعلى
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

# ابدأ التدريب فوراً!
trainer_stats = trainer.train()

In [ ]:
# 6. حفظ الموديل النهائي بصيغة GGUF (النسخة التي تعمل محلياً على Ollama)
model.save_pretrained_gguf("model_haitham_accountant", tokenizer, quantization_method = "q4_k_m")
print("✅ تم حفظ الموديل بنجاح!")